# Module 10: Important Modules

**Utrains Python Fundamentals** &middot; lab notebook

*Import the standard library, set up a virtual environment, and meet openai, langchain, and langgraph.*

## What you will be able to do by the end

- Say what a module is and what import actually does
- Use all three import patterns
- Measure time and read the operating system with time and os
- Keep secrets out of your code and out of source control
- Say what openai, langchain and langgraph each do, and when to pick which

## How this notebook is organised

It follows the Module 10 slide deck, slide by slide.

Run every cell in order with **Shift + Enter**. Read the note above each block,
then run the code and compare what you see with what you expected.

Two cells are marked **Your turn**. They contain `____` where a piece of the
syntax is missing, so they will fail if you run them as they are. That is
deliberate. Replace each `____`, then run the cell until it succeeds.

The last section is the **Lab**: a short task with no code written for you.

## What this notebook assumes

Modules 1 to 9. The AI package slides need network access and an API key, so their code is shown as reference and paired with a standard library stand-in you can actually run.

## Slide 2 &middot; What Is a Module?

A module is a file of pre-written Python code you bring into your own program
with `import`. This is where most of Python's real power comes from: you rarely
have to write something from scratch.

**Standard library vs third party.** `time` ships with Python. Something like
`ollama` does not, so you install it first.

In [ ]:
import time            # standard library, comes with Python

# import ollama       # third party, pip install ollama first

print("time module loaded:", time.__name__)

## Slide 3 &middot; Import Patterns

`import` loads a module so its functions and variables become available. There
are a few common styles.

| Pattern | Gives you |
|---|---|
| `import module_name` | `module_name.something` |
| `import module_name as alias` | a shorter name |
| `from module_name import something` | just one piece, directly |

In [ ]:
import datetime as dt
from math import sqrt

print(dt.date.today())
print(sqrt(16))

## Slide 4 &middot; Measuring Time

The `time` module is the standard way to measure how long something takes to
run. `time.time()` gives you a number of seconds; take one from another to get
the elapsed time.

In [ ]:
import time

t0 = time.time()
time.sleep(0.3)
print(f"Waited {time.time() - t0:.1f}s")

---

### Your turn 1

Measure how long a fake health check takes and print the answer to two decimal places. Two ideas here: the import statement, and the call that reads the clock.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
# TODO: bring in the module that can pause and read the clock.
____ time


def run_healthcheck():
    time.sleep(0.25)
    return "healthy"


# TODO: read the clock before and after.
start = time.____()
status = run_healthcheck()
elapsed = time.____() - start

print(f"health check returned {status} in {elapsed:.2f}s")

## Slide 5 &middot; Working with the Operating System

The `os` module reads information about the machine your script is running on.

In [ ]:
import os

print(os.getcwd())                      # current directory
print(sorted(os.listdir("."))[:10])     # files here

## Slide 6 &middot; Config Values versus Secrets

Config is safe to keep in your code. Secrets, like API keys, should never be
committed to source control. Load them from a `.env` file instead.

```python
from dotenv import load_dotenv
import os

load_dotenv()

MODEL = "claude-sonnet-4-6"                    # config, fine to commit
api_key = os.environ.get("ANTHROPIC_API_KEY")  # secret, never commit

print("API key loaded:", bool(api_key))
```

The `.env` file, kept out of version control by `.gitignore`:

```
ANTHROPIC_API_KEY=sk-...
```

**Never print the raw key, even in a demo.** The cell below does the same job
with only the standard library, so it runs without installing anything, and it
prints whether the key exists rather than the key itself.

In [ ]:
import os

MODEL = "claude-sonnet-4-6"
api_key = os.environ.get("ANTHROPIC_API_KEY")

print("model:", MODEL)
print("API key loaded:", bool(api_key))

## Slide 7 &middot; API, SDK, and Client, Defined

Three words you will see constantly once you start calling AI models.

- **API** &mdash; the remote interface your program talks to over the network
- **SDK** &mdash; the library you import to talk to that API without writing raw
  HTTP calls, such as `anthropic` or `openai`
- **client** &mdash; the object the SDK gives you to actually make calls, for
  example `client = Anthropic()`

```python
# from anthropic import Anthropic
# client = Anthropic()
# r = client.messages.create(model=..., messages=...)
```

## Slide 8 &middot; Setting Up a Virtual Environment

The packages coming up do not ship with Python. Before installing anything,
isolate this project's packages from the rest of your system.

**One environment per project.** This keeps one project's packages from
colliding with another's, and lets you delete `.venv` and start clean any time.

```bash
uv venv                        # creates .venv here

source .venv/bin/activate      # Linux / macOS
.venv\Scripts\activate         # Windows

uv pip install openai langchain langgraph

deactivate                     # leave when you are done
```

If you followed this repo's README, you already did exactly this to get the
notebook running. The `README.md` has the full walkthrough including how to
point Jupyter at the environment.

## Slide 9 &middot; Popular AI Packages: openai

The official SDK for calling OpenAI's models. A client object handles the
network request, and you read the reply off the response object.

Installed inside your venv with `uv pip install openai`.

```python
from openai import OpenAI

client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say hello."}],
)
print(response.choices[0].message.content)
```

## Slide 10 &middot; Popular AI Packages: langchain

LangChain wraps different model providers behind a common interface, so the
same code can call OpenAI, Anthropic or others with only the client changed.

**Same shape, different provider.** Swap `ChatOpenAI` for a different
provider's class and the rest of the code barely changes.

```python
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")
response = llm.invoke([HumanMessage(content="Say hello.")])
print(response.content)
```

## Slide 11 &middot; Popular AI Packages: langgraph

LangGraph structures an AI program as a graph of steps, called **nodes**,
connected by **edges**. Each node is a plain Python function.

**State flows through the graph.** Each node receives the current state and
returns an update to it, one step at a time.

```python
from langgraph.graph import StateGraph, END

def greet_node(state):
    return {"message": f"Hello, {state['name']}!"}

graph = StateGraph(dict)
graph.add_node("greet", greet_node)
graph.set_entry_point("greet")
graph.add_edge("greet", END)

app = graph.compile()
result = app.invoke({"name": "Alice"})
print(result["message"])       # Hello, Alice!
```

The cell below is that same graph with nothing but the standard library. The
point is the shape: a node is a function, and state flows through it.

In [ ]:
def greet_node(state):
    return {"message": f"Hello, {state['name']}!"}


def run_graph(nodes, state):
    for node in nodes:
        state.update(node(state))
    return state


result = run_graph([greet_node], {"name": "Alice"})
print(result["message"])

## Slide 12 &middot; LangChain Agents vs LangGraph

Both can build an **agent**, a program that decides what to do rather than
always running fixed steps. They hand you different amounts of control.

**LangChain agents.** Faster to set up for a standard tool-calling loop. Hand
it a model and tools, and it repeats until done.

**LangGraph.** More code to wire up, but you see and control every step. Add
cycles, or pause for human approval.

**No wrong choice.** Reach for LangChain's agent tools for a quick, standard
loop. Reach for LangGraph when you need to see and control the steps yourself.
Complex or multi-agent workflows tend to move toward LangGraph as they grow.

## Slide 13 &middot; A LangChain Agent in Practice

This is the newer, tool-calling style of LangChain agent: hand it a model, a
list of tools and a prompt.

**`@tool` marks a function** as something the model is allowed to call. The
docstring tells the model what the tool does.

```python
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate

@tool
def get_weather(city: str) -> str:
    """Look up the weather for a city."""
    return f"It is sunny in {city}."

agent = create_tool_calling_agent(llm, [get_weather], prompt)
executor = AgentExecutor(agent=agent, tools=[get_weather])
result = executor.invoke({"input": "Weather in Boston?"})
print(result["output"])
```

## Slide 14 &middot; Building a Tiny Agent with LangGraph

A small agent, built with the same `StateGraph` pattern, that answers weather
questions and declines anything else.

**This is what every agent does.** Read the state, decide what to do, return an
update. The `if` / `else` here stands in for a model deciding which tool to
call.

In [ ]:
def get_weather(city):
    # pretend this calls a real weather API
    return f"It is sunny in {city}."


def agent_node(state):
    question = state["question"]
    if "weather" in question.lower():
        answer = get_weather("Boston")
    else:
        answer = "I can only answer weather questions."
    return {"answer": answer}


print(agent_node({"question": "What is the weather like?"})["answer"])
print(agent_node({"question": "Who won the game?"})["answer"])

---

### Your turn 2

Extend the tiny agent with a second condition so it also answers a simple addition question, and still declines everything else.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
def agent_node(state):
    question = state["question"].lower()

    if "weather" in question:
        answer = get_weather("Boston")
    # TODO: add a branch that catches a maths question, then a catch-all.
    ____ "plus" ____ question:
        answer = "That is 4."
    ____:
        answer = "I can only answer weather or simple maths right now."

    return {"answer": answer}


for q in ["What is the weather like?", "what is 2 plus 2", "Who won the game?"]:
    print(q, "->", agent_node({"question": q})["answer"])

## Slide 15 &middot; Other Useful Modules

A quick map of what else is out there, for when you need it.

- **subprocess** &mdash; running shell commands from inside Python
- **requests** &mdash; making HTTP calls to APIs
- **Docker and Kubernetes** &mdash; container and orchestration helper patterns
- **boto3 and Terraform** &mdash; cloud and infrastructure as code
- **Git automation** &mdash; CI/CD pipeline integration

---

## Lab: A timed, config-driven health reporter


Write a small script in this notebook that does four things.

Read a model name from an environment variable called `LAB_MODEL`, falling back
to `"gpt-4o-mini"` when it is not set. Print the model, and separately print
whether an API key called `LAB_API_KEY` is present, without ever printing its
value.

Use `os.listdir` to count how many files sit in the current folder.

Time a fake `run_healthcheck()` function that sleeps briefly and returns a
status, and print the elapsed time to two decimal places.

Stamp the report with today's date using `datetime`.

Print all of it as one tidy report block.


**Done when:**

- [ ] The model comes from the environment with a fallback
- [ ] The key is reported as present or absent, never printed
- [ ] os is used to count the files in the folder
- [ ] time measures the health check, datetime stamps the report

Write your answer in the cell below. There is no starter code on purpose.

In [ ]:
# Your lab answer goes here.

---

## Practice exercises

These are the four exercises from the module's practice slide, word for word.

1. Use the os module to list every file in the current deployment directory.
2. Use dotenv to load a cloud provider's API key from a .env file, printing only whether it loaded, never the key itself.
3. Use the time module to measure how long a fake health check function takes to run.
4. Extend the tiny agent above with a second condition, so it also answers a simple math question like "what is 2 plus 2".

---

## Module complete

You can now import modules, manage secrets, and build with real AI packages. Exercise 2 needs `python-dotenv`, which is in `requirements-ai.txt`.

*Utrains &middot; support@utrains.org &middot; https://utrains.org*